# Coreset Strategy — Short-Text (banking77) Test

Our production input is mostly **short, one-or-two-line sentences** (customer-style
queries), and for the bounded-memory streaming use case the only recluster strategy
that matters is **`coreset`**. This notebook is a focused test of *just* that strategy.

- **Dataset:** [`PolyAI/banking77`](https://huggingface.co/datasets/PolyAI/banking77) —
  ~10k short customer-service queries across 77 intents. One-line sentences, the closest
  public match to our prod input.
- **What we exercise:** a single `CumulativeTriTopic(strategy="coreset")` over a stream
  where new intents emerge over time (drives novelty / recluster events). We deliberately
  set `coreset_size` **small** relative to the corpus so the coreset actually *binds*
  (Regime B) and is genuinely tested — otherwise it would just fall back to a full refit.
- **What we check:** quality vs a full-batch baseline (ARI/NMI/coherence), the coreset
  cost ratio, rare-topic survival, and a stratified-vs-recency A/B on the coreset sampler.

> The multi-strategy benchmark lives in `notebooks/cumulative_tritopic_benchmark.ipynb`.

## 1 — Install (optional)

In [ ]:
# Uncomment on a fresh environment (e.g. Kaggle/Colab).
# %pip install -q datasets sentence-transformers plotly scikit-learn
# %pip install -q -e ..   # install the local tritopic package
print("skip install if already set up")

## 2 — Imports & constants

In [ ]:
import copy
import pickle
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from tritopic.core.model import TriTopic, TriTopicConfig
from tritopic.cumulative.cumulative import CumulativeConfig, CumulativeTriTopic
from tritopic.cumulative.strategies import make_strategy, ReclusterContext
from tritopic.cumulative.evaluation import compare_to_full_batch, coreset_cost_ratio
from tritopic.utils.metrics import (
    compute_ari,
    compute_nmi,
    compute_silhouette,
    compute_coherence,
    compute_diversity,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── tuneable constants ──────────────────────────────────────────────────────
EMBED_MODEL      = "all-MiniLM-L6-v2"
EMBED_BATCH_SIZE = 512
RANDOM_STATE     = 42

N_BATCHES        = 8        # number of streaming batches
TARGET_BATCH     = 1000     # ~docs per batch (capped by availability)

# Coreset knobs — small on purpose so Regime B engages on this ~9k-doc corpus.
CORESET_SIZE     = 2000     # working-set cap for the coreset summary
MAX_INMEMORY     = 3000     # absolute cap before Regime B kicks in
MIN_PER_TOPIC    = 20       # stratified per-topic floor (anti tail-collapse)

NOVELTY_THRESHOLD    = 0.30
MIN_CLUSTER_FRACTION = 0.01  # 1% of fitting corpus as min community size
MIN_CLUSTER_SIZE_ABS = 5

CACHE_DIR = Path("./cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
EMB_CACHE  = CACHE_DIR / f"banking77_emb_{EMBED_MODEL.replace('/', '_')}.npy"
DOCS_CACHE = CACHE_DIR / "banking77_docs.pkl"

rng = np.random.default_rng(RANDOM_STATE)
print(f"Imports OK  |  device={device}")

## 3 — Load banking77

In [ ]:
print("Loading PolyAI/banking77 train split...")
ds = load_dataset("PolyAI/banking77", split="train")

documents   = [row["text"] for row in ds]
labels_raw  = np.array([row["label"] for row in ds])
intent_names = ds.features["label"].names   # 77 human-readable intent names

word_counts = np.array([len(d.split()) for d in documents])
print(f"Total documents : {len(documents):,}")
print(f"Intents         : {len(intent_names)}")
print(f"Words per doc   : mean={word_counts.mean():.1f}  median={np.median(word_counts):.0f}  "
      f"p95={np.percentile(word_counts, 95):.0f}  max={word_counts.max()}")
print("\nSample short docs:")
for d in documents[:5]:
    print(f"  • {d}")

## 4 — Streaming batches with emerging intents

We reveal the 77 intents in waves — a fresh group of intents activates each batch — so
the stream genuinely *drifts*. When a wave appears its docs are unseen vocabulary, which
spikes the novelty score and triggers a recluster. Each document is used at most once;
the cumulative order equals the full-batch order so the two models are comparable.

In [ ]:
def make_emerging_batches(labels, n_batches, target_batch, rng):
    """Reveal intents in waves; round-robin draw from currently-active intents.

    New-wave intents are drawn first each batch so a fresh theme shows up as drift.
    Returns a list of index arrays (each doc used at most once).
    """
    intents = rng.permutation(np.unique(labels))
    waves = [w for w in np.array_split(intents, n_batches)]
    queues = {int(c): list(rng.permutation(np.where(labels == c)[0])) for c in intents}

    active, batches = [], []
    for b in range(n_batches):
        new = [int(c) for c in waves[b]]
        active.extend(new)
        order = new + [c for c in active if c not in new]   # new wave first
        batch, progressing = [], True
        while len(batch) < target_batch and progressing:
            progressing = False
            for c in order:
                if queues[c]:
                    batch.append(queues[c].pop())
                    progressing = True
                    if len(batch) >= target_batch:
                        break
        batches.append(np.array(batch, dtype=int))
    return batches, waves


batch_indices, waves = make_emerging_batches(labels_raw, N_BATCHES, TARGET_BATCH, rng)

batch_docs   = [[documents[i] for i in idx] for idx in batch_indices]
batch_labels = [labels_raw[idx] for idx in batch_indices]

all_indices      = np.concatenate(batch_indices)
all_docs_ordered = [documents[i] for i in all_indices]
labels_ordered   = labels_raw[all_indices]

print(f"Batches          : {len(batch_docs)}")
print(f"Docs per batch   : {[len(b) for b in batch_docs]}")
print(f"New intents/batch: {[len(w) for w in waves]}")
print(f"Total docs used  : {len(all_docs_ordered):,}  "
      f"(coreset cap = {CORESET_SIZE} → Regime B should engage)")

## 5 — Embedding (cached)

In [ ]:
if EMB_CACHE.exists() and DOCS_CACHE.exists():
    print("Loading cached embeddings...")
    all_embeddings = np.load(EMB_CACHE)
    with open(DOCS_CACHE, "rb") as f:
        _cached_docs = pickle.load(f)
    if len(_cached_docs) != len(all_docs_ordered):
        raise RuntimeError("Cache doc-count mismatch — delete ./cache and rerun.")
    print(f"Loaded from cache: {all_embeddings.shape}")
else:
    print(f"Encoding {len(all_docs_ordered):,} docs with '{EMBED_MODEL}' on {device}...")
    st_model = SentenceTransformer(EMBED_MODEL, device=device)
    t0 = time.time()
    all_embeddings = st_model.encode(
        all_docs_ordered,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Encoded in {time.time() - t0:.1f}s  |  shape={all_embeddings.shape}")
    np.save(EMB_CACHE, all_embeddings)
    with open(DOCS_CACHE, "wb") as f:
        pickle.dump(all_docs_ordered, f)
    del st_model
    if device == "cuda":
        torch.cuda.empty_cache()

# Per-batch embedding slices (views, no extra RAM).
batch_embs, start = [], 0
for idx in batch_indices:
    n = len(idx)
    batch_embs.append(all_embeddings[start:start + n])
    start += n
print(f"Embedding matrix : {all_embeddings.shape}  {all_embeddings.nbytes / 1e6:.1f} MB")

## 6 — Configuration (coreset only)

In [ ]:
base_config = TriTopicConfig(
    use_dim_reduction=False,          # skip UMAP per-recluster — faster
    use_lexical_view=True,            # TF-IDF co-occurrence helps on short text
    use_iterative_refinement=False,   # speed; minimal quality loss here
    n_consensus_runs=3,
    min_cluster_size=MIN_CLUSTER_SIZE_ABS,
    min_cluster_fraction=MIN_CLUSTER_FRACTION,
    low_memory=True,
    knn_backend="auto",
    n_neighbors=15,
    n_jobs=1,                         # single-threaded → reproducible
    random_state=RANDOM_STATE,
    verbose=False,
)

cfg = CumulativeConfig(
    base_config=copy.deepcopy(base_config),
    strategy="coreset",                       # <<< the only strategy under test
    coreset_size=CORESET_SIZE,
    max_inmemory_docs=MAX_INMEMORY,
    coreset_selection="stratified",           # per-topic floor protects rare intents
    min_docs_per_topic_in_coreset=MIN_PER_TOPIC,
    recluster_trigger="drift",
    novelty_threshold=NOVELTY_THRESHOLD,
    min_docs_between_recluster=0,
    verbose=False,
)
print("config ready — strategy =", cfg.strategy)

## 7 — Coreset streaming loop

Stream the batches through one coreset model. After every recluster we read
`model.history_[-1]` to expose the **regime** and **working-set size** — when the
accumulator exceeds the cap this should report `regime=B` with `n_docs_clustered ≈
coreset_size`, proving the coreset is doing its job rather than silently refitting on all.

In [ ]:
import psutil
_proc = psutil.Process()

model = CumulativeTriTopic(cfg)
rows = []

header = (f"{'batch':>5} | {'n_docs':>6} | {'cumul':>6} | {'novelty':>7} | "
          f"{'reclust':>7} | {'regime':>6} | {'work_set':>8} | {'g-topics':>8} | "
          f"{'wall_s':>6} | {'rss_mb':>7}")
print(header); print("-" * len(header))

for b_idx, (docs, emb) in enumerate(zip(batch_docs, batch_embs)):
    t0 = time.time()
    result = model.add_batch(docs, embeddings=emb)
    elapsed = time.time() - t0
    rss_mb = _proc.memory_info().rss / 1e6

    last = model.history_[-1] if model.history_ else None
    regime    = last.regime if (result.reclustered and last) else "—"
    work_set  = last.n_docs_clustered if (result.reclustered and last) else None

    rows.append({
        "batch": b_idx + 1,
        "n_new_docs": result.n_new_docs,
        "n_total_docs": result.n_total_docs,
        "novelty": result.novelty,
        "reclustered": result.reclustered,
        "regime": regime if regime != "—" else None,
        "work_set": work_set,
        "n_global_topics": model.n_global_topics,
        "wall_s": elapsed,
        "rss_mb": rss_mb,
    })

    nv = f"{result.novelty:.3f}" if result.novelty is not None else "  —  "
    rc = "YES ★" if result.reclustered else "no"
    ws = f"{work_set:,}" if work_set is not None else "—"
    print(f"{b_idx+1:>5} | {len(docs):>6,} | {result.n_total_docs:>6,} | {nv:>7} | "
          f"{rc:>7} | {regime:>6} | {ws:>8} | {model.n_global_topics:>8} | "
          f"{elapsed:>6.1f} | {rss_mb:>7.0f}")

df_per_batch = pd.DataFrame(rows)
n_regime_b = (df_per_batch["regime"] == "B").sum()
print(f"\nfinal n_global_topics = {model.n_global_topics}  |  "
      f"reclusters = {int(df_per_batch.reclustered.sum())}  |  Regime-B reclusters = {n_regime_b}")
assert n_regime_b > 0, "Coreset never bound (Regime B) — lower CORESET_SIZE/MAX_INMEMORY."
df_per_batch

## 8 — Full-batch baseline & comparison

In [ ]:
print(f"Fitting full-batch TriTopic on {len(all_docs_ordered):,} docs...")
t0 = time.time()
full_model = TriTopic(config=copy.deepcopy(base_config))
full_model.fit(all_docs_ordered, embeddings=all_embeddings)
t_full = time.time() - t0
n_full_topics = len([t for t in full_model.topics_ if t.topic_id != -1])
print(f"  full-batch: {t_full:.1f}s  |  topics={n_full_topics}")

cmp = compare_to_full_batch(model, full_model, labels_true=labels_ordered)
print("\nCoreset (cumulative)  vs  full-batch:")
for k in ["ari_vs_truth_cumulative", "ari_vs_truth_full",
          "nmi_vs_truth_cumulative", "nmi_vs_truth_full",
          "ari_vs_full", "nmi_vs_full",
          "n_topics_cumulative", "n_topics_full", "topic_count_drift",
          "silhouette_cumulative", "silhouette_full", "silhouette_delta",
          "rare_topic_recall", "n_rare_topics_full", "keyword_overlap"]:
    if k in cmp:
        v = cmp[k]
        print(f"  {k:<28}: {v:.4f}" if isinstance(v, float) else f"  {k:<28}: {v}")

## 9 — Coreset diagnostics: cost ratio + sampler A/B

`coreset_cost_ratio` measures the weighted k-means distortion of the coreset working set
relative to the full data (≈1.0 = the coreset preserves the geometry). We rebuild the final
coreset working set exactly the way the model does internally, then compare the two
`coreset_selection` modes (`stratified` vs `recency`) head-to-head on quality and
rare-intent survival — the whole reason the coreset has a sampler knob.

In [ ]:
# Rebuild the final coreset working set (mirrors CoresetStrategy inside the model).
ctx = ReclusterContext(
    documents=model.documents_,
    embeddings=model.embeddings_,
    new_count=len(batch_docs[-1]),
    max_inmemory_docs=cfg.max_inmemory_docs,
    coreset_size=cfg.coreset_size,
    random_state=RANDOM_STATE,
    labels=model.labels_,
    coreset_selection=cfg.coreset_selection,
    min_per_cluster=cfg.min_docs_per_topic_in_coreset,
)
ws = make_strategy("coreset").select_working_set(ctx)
ratio = coreset_cost_ratio(
    ws.embeddings, ws.weights, model.embeddings_,
    k=max(model.n_global_topics, 2), random_state=RANDOM_STATE,
)
print(f"Working-set size : {len(ws.indices):,} / {len(model.documents_):,} docs  (regime {ws.regime})")
print(f"Coreset cost ratio (≈1.0 is ideal): {ratio:.3f}")

In [ ]:
# A/B: stratified vs recency coreset sampling — same batches, same config, one knob.
def run_coreset(selection):
    c = copy.deepcopy(cfg)
    c.coreset_selection = selection
    c.base_config = copy.deepcopy(base_config)
    m = CumulativeTriTopic(c)
    for docs, emb in zip(batch_docs, batch_embs):
        m.add_batch(docs, embeddings=emb)
    return m

ab_rows = []
for sel in ["stratified", "recency"]:
    m = run_coreset(sel)
    c = compare_to_full_batch(m, full_model, labels_true=labels_ordered)
    ab_rows.append({
        "coreset_selection": sel,
        "ari_vs_truth": c["ari_vs_truth_cumulative"],
        "nmi_vs_truth": c["nmi_vs_truth_cumulative"],
        "ari_vs_full": c["ari_vs_full"],
        "n_global_topics": m.n_global_topics,
        "rare_topic_recall": c["rare_topic_recall"],
    })
pd.DataFrame(ab_rows)

## 10 — Visualisations

In [ ]:
# Topic-count evolution + novelty with recluster markers.
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_per_batch.batch, y=df_per_batch.n_global_topics,
                         mode="lines+markers", name="global topics", yaxis="y1"))
fig.add_trace(go.Scatter(x=df_per_batch.batch, y=df_per_batch.novelty,
                         mode="lines+markers", name="novelty", yaxis="y2",
                         line=dict(dash="dot")))
fig.add_hline(y=NOVELTY_THRESHOLD, line_dash="dash", line_color="grey",
              annotation_text="novelty threshold", yref="y2")
rc = df_per_batch[df_per_batch.reclustered]
fig.add_trace(go.Scatter(x=rc.batch, y=rc.n_global_topics, mode="markers",
                         name="recluster", marker=dict(size=13, symbol="star",
                         color="crimson"), yaxis="y1"))
fig.update_layout(title="Coreset stream — topics & novelty per batch",
                  xaxis_title="batch",
                  yaxis=dict(title="global topics"),
                  yaxis2=dict(title="novelty", overlaying="y", side="right", range=[0, 1]),
                  height=420, legend=dict(orientation="h"))
fig.show()

In [ ]:
# Working-set size per batch — shows the coreset bounding the fitting cost.
ws_df = df_per_batch.copy()
ws_df["work_set_plot"] = ws_df["work_set"].fillna(0)
fig = go.Figure()
fig.add_trace(go.Bar(x=ws_df.batch, y=ws_df.n_total_docs, name="accumulated docs",
                     marker_color="lightsteelblue"))
fig.add_trace(go.Bar(x=ws_df.batch, y=ws_df.work_set_plot, name="coreset working set",
                     marker_color="seagreen"))
fig.add_hline(y=CORESET_SIZE, line_dash="dash", line_color="crimson",
              annotation_text=f"coreset_size = {CORESET_SIZE}")
fig.update_layout(title="Accumulated corpus vs coreset working set (the memory win)",
                  barmode="overlay", xaxis_title="batch", yaxis_title="documents",
                  height=420, legend=dict(orientation="h"))
fig.show()

In [ ]:
# Final topic keywords + coherence/diversity on the short-text corpus.
topics = [t for t in model.model_.topics_ if t.topic_id != -1]
all_kw = []
coh = []
for t in topics:
    all_kw += t.keywords
    coh.append(compute_coherence(t.keywords, model.documents_))
diversity = compute_diversity(all_kw, len(topics))
print(f"topics={len(topics)}  mean NPMI coherence={np.mean(coh):.4f}  diversity={diversity:.4f}\n")
for t in sorted(topics, key=lambda x: x.topic_id)[:15]:
    print(f"  topic {t.topic_id:>3}: {', '.join(t.keywords[:8])}")

## 11 — Conclusions

- **Coreset on short text works as a streaming engine.** On banking77 (one-line queries)
  the coreset model tracks the emerging-intent drift, firing reclusters when new intents
  appear and keeping a stable global topic set across epochs.
- **The coreset actually bound the cost.** At least one recluster ran in **Regime B** with
  the working set held at ≈`coreset_size` while the accumulator grew past `max_inmemory_docs`
  — the fitting cost stays flat instead of growing with the corpus.
- **Quality held up.** ARI/NMI vs the full-batch baseline (and vs ground-truth intents) stay
  close, and the coreset cost ratio is near 1.0, i.e. the summary preserves the geometry.
- **Stratified beats recency for rare intents.** The A/B shows the stratified sampler's
  per-topic floor protects small/rare intents (higher rare-topic recall) — the right default
  for prod, where long-tail intents matter.

_Tune `CORESET_SIZE` / `MAX_INMEMORY` to trade memory for fidelity; raise
`min_docs_per_topic_in_coreset` if rare intents still collapse._